# Padding the chunks (SANEPIC's "inpaint")

pysanepic applies N⁻¹ with FFTs on chunks of `chunk_s` seconds. An FFT treats
each chunk as periodic: its end is joined to its start, and with 1/f noise the
two ends are at different levels, so N⁻¹ sees an artificial jump and the
noise correlations at lags comparable to the chunk length are wrong.

With `pad_s > 0` every chunk is extended by `pad_s` seconds on both sides, so
that the wrap-around falls in the padding. Each margin points to its own
*virtual pixel*, an offset fitted by the map-maker and not included in the
output maps. Two fills are available:

* `"zeros"` (default): exactly described by the margin offsets, so it adds no
  bias;
* `"extrapolate"` (SANEPIC's choice): linear extrapolation of the chunk ends,
  tapered to the chunk mean. Its shape is not a constant, so a small part of it
  leaks into the map through the 1/f weighting.

This notebook measures the effect on one day of LiteBIRD without HWP, with the
1/f noise of the IMO and with a much steeper 1/f noise.

Requirements: `pysanepic` and a `litebird_sim` version providing
`Simulation.make_sanepic_gls_map` (until [litebird_sim#568](https://github.com/litebird/litebird_sim/pull/568)
is merged: `pip install git+https://github.com/litebird/litebird_sim@sanepic_gls`).

In [ ]:
import time

import astropy.time
import healpy as hp
import litebird_sim as lbs
import numpy as np
from pysanepic import GLSParameters

In [ ]:
telescope, channel = "LFT", "L4-140"
# Q-type and U-type T/B pairs: without HWP both are needed to separate Q and U
detlist = ["000_001_016_QA_140_T", "000_001_016_QA_140_B", "000_001_014_UA_140_T", "000_001_014_UA_140_B"]
duration_days = 1.0
nside = 64
imo = lbs.Imo(flatfile_location=lbs.PTEP_IMO_LOCATION)


def simulate(fknee_mhz=None, alpha=None):
    sim = lbs.Simulation(
        base_path="padding_output",
        imo=imo,
        start_time=astropy.time.Time("2030-01-01T00:00:00"),
        duration_s=duration_days * 86400,
        random_seed=12345,
    )
    sim.set_instrument(
        lbs.InstrumentInfo.from_imo(imo, f"/releases/vPTEP/satellite/{telescope}/instrument_info")
    )
    dets = [
        lbs.DetectorInfo.from_imo(
            url=f"/releases/vPTEP/satellite/{telescope}/{channel}/{d}/detector_info", imo=imo
        )
        for d in detlist
    ]
    for d in dets:  # optionally override the 1/f parameters of the IMO
        d.fknee_mhz = fknee_mhz or d.fknee_mhz
        d.alpha = alpha or d.alpha
    sim.set_scanning_strategy(imo_url="/releases/vPTEP/satellite/scanning_parameters/")
    sim.create_observations(detectors=dets)
    sim.prepare_pointings()  # no HWP
    ch = lbs.FreqChannelInfo.from_imo(
        url=f"/releases/vPTEP/satellite/{telescope}/{channel}/channel_info", imo=imo
    )
    sim.get_sky(
        parameters=lbs.SkyGenerationParams(
            make_cmb=True, make_fg=False, seed_cmb=1, apply_beam=True,
            units="K_CMB", output_type="map", nside=nside,
        ),
        channels=[ch],
        store_in_observation=True,
    )
    sim.fill_tods()
    # noiseless reference map
    reference = sim.make_binned_map(nside=nside).binned_map
    sim.add_noise(noise_type="one_over_f")
    return sim, reference

## Map-making with different chunks and paddings

The residual is the map minus the binned map of the noiseless signal (I monopole
removed). Without padding, the longer the chunk, the larger the jump between its
two ends, because 1/f noise drifts.

In [ ]:
runs = {
    "1 chunk (whole day)": GLSParameters(chunk_s=duration_days * 86400, maxiter=3000),
    "1 chunk + pad 3600 s, zeros": GLSParameters(chunk_s=duration_days * 86400, pad_s=3600.0, pad_fill="zeros", maxiter=3000),
    "1 chunk + pad 3600 s, extrapolate": GLSParameters(chunk_s=duration_days * 86400, pad_s=3600.0, pad_fill="extrapolate", maxiter=3000),
    "chunk 3600 s": GLSParameters(chunk_s=3600.0, maxiter=3000),
    "chunk 3600 s + pad 900 s, zeros": GLSParameters(chunk_s=3600.0, pad_s=900.0, pad_fill="zeros", maxiter=3000),
}


def residual_rms(res, reference):
    seen = (res.maps[1] != hp.UNSEEN) & np.isfinite(reference[1])
    r = res.maps[:, seen] - reference[:, seen]
    r[0] -= r[0].mean()
    return 1e6 * r.std(axis=1)


for title, noise in [("IMO 1/f noise", {}), ("steep 1/f noise: fknee = 100 mHz, alpha = 2", dict(fknee_mhz=100.0, alpha=2.0))]:
    sim, reference = simulate(**noise)
    obs = sim.observations[0]
    print(f"{title} (fknee = {obs.fknee_mhz[0]} mHz, alpha = {obs.alpha[0]})")
    for name, params in runs.items():
        t0 = time.time()
        res = sim.make_sanepic_gls_map(nside=nside, params=params, append_to_report=False)
        rms = residual_rms(res, reference)
        print(f"  {name:38s} {res.iterations:4d} iterations {time.time() - t0:6.1f} s   "
              f"rms residual I/Q/U [uK]: " + " / ".join(f"{x:6.2f}" for x in rms))

## Summary

rms residual I / Q / U [µK] on this day of data:

| configuration | IMO 1/f (f_knee 20 mHz, α 1) | steep 1/f (f_knee 100 mHz, α 2) |
|---|---|---|
| 1 chunk (whole day) | 16.55 / 23.99 / 24.23 | 79.48 / 67.78 / 97.29 |
| 1 chunk + pad 3600 s, zeros | 16.55 / 23.99 / 24.23 | 41.59 / 50.16 / 61.77 |
| 1 chunk + pad 3600 s, extrapolate | 16.55 / 23.99 / 24.23 | 32.68 / 44.60 / 49.13 |
| chunk 3600 s | 16.55 / 23.99 / 24.23 | 36.09 / 55.81 / 63.52 |
| chunk 3600 s + pad 900 s, zeros | 16.55 / 23.99 / 24.22 | 33.71 / 49.67 / 55.92 |

* With the IMO noise the edges of the chunks do not matter: chunks of one hour
  or more give the same maps with or without padding.
* With a steep 1/f noise the data drift, the two ends of a chunk are far apart
  and the FFT wrap-around degrades the maps, the more the longer the chunk.
  Padding removes most of this; `"extrapolate"`, which continues the drift,
  works best here.
* With short chunks and a mild 1/f noise `"extrapolate"` can bias I (its shape
  is not a constant); `"zeros"` is unbiased, hence the default.